In [9]:
import pandas as pd
df = pd.read_csv('cs-training.csv')
df.head()

,Unnamed: 0,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
1,2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
2,3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
3,4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
4,5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


In [10]:
df.shape

(150000, 12)

In [11]:
df = df.drop(columns='Unnamed: 0')
df.head()

,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
1,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
2,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
3,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
4,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


In [12]:
df.shape

(150000, 11)

In [13]:
df.isnull().sum()

SeriousDlqin2yrs                            0
RevolvingUtilizationOfUnsecuredLines        0
age                                         0
NumberOfTime30-59DaysPastDueNotWorse        0
DebtRatio                                   0
MonthlyIncome                           29731
NumberOfOpenCreditLinesAndLoans             0
NumberOfTimes90DaysLate                     0
NumberRealEstateLoansOrLines                0
NumberOfTime60-89DaysPastDueNotWorse        0
NumberOfDependents                       3924
dtype: int64

In [14]:
X = df.drop(columns='SeriousDlqin2yrs')
y = df['SeriousDlqin2yrs']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [15]:
X_train.shape, X_test.shape

((120000, 10), (30000, 10))

In [16]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

In [17]:
import numpy as np
np.isnan(X_train).sum()

np.int64(0)

In [18]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
dummy_preds = dummy.predict(X_test)

accuracy_score(y_test, dummy_preds)

0.9331666666666667

In [19]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)
logreg_preds = logreg.predict(X_test)

accuracy_score(y_test, logreg_preds)

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0.9339333333333333

In [20]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)
logreg_preds = logreg.predict(X_test)

accuracy_score(y_test, logreg_preds)

0.934

In [21]:
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

logreg_probs = logreg.predict_proba(X_test)[:,1]

print("ROC-AUC", roc_auc_score(y_test, logreg_probs))
print()
print(confusion_matrix(y_test, logreg_preds))
print()
print(classification_report(y_test, logreg_preds))

ROC-AUC 0.7143280217031988

[[27930    65]
 [ 1915    90]]

              precision    recall  f1-score   support

           0       0.94      1.00      0.97     27995
           1       0.58      0.04      0.08      2005

    accuracy                           0.93     30000
   macro avg       0.76      0.52      0.52     30000
weighted avg       0.91      0.93      0.91     30000



In [22]:
threshold = 0.2
logreg_preds_low = (logreg_probs >= threshold).astype(int)

print(confusion_matrix(y_test, logreg_preds_low))
print()
print(classification_report(y_test,logreg_preds_low))

[[27561   434]
 [ 1663   342]]

              precision    recall  f1-score   support

           0       0.94      0.98      0.96     27995
           1       0.44      0.17      0.25      2005

    accuracy                           0.93     30000
   macro avg       0.69      0.58      0.60     30000
weighted avg       0.91      0.93      0.92     30000



In [23]:
for threshold in [0.50, 0.30, 0.20, 0.15, 0.10, 0.05]:
    preds = (logreg_probs >= threshold).astype(int)
    cm = confusion_matrix(y_test, preds)
    caught = cm[1,1]
    missed = cm[1,0]
    false_alarms = cm[0,1]
    print(f"threshold {threshold:.2f}:  caught {caught:4d} / 2005 defaulters   |   missed {missed:4d}   |   false alarms {false_alarms:5d}")

threshold 0.50:  caught   90 / 2005 defaulters   |   missed 1915   |   false alarms    65
threshold 0.30:  caught  204 / 2005 defaulters   |   missed 1801   |   false alarms   175
threshold 0.20:  caught  342 / 2005 defaulters   |   missed 1663   |   false alarms   434
threshold 0.15:  caught  517 / 2005 defaulters   |   missed 1488   |   false alarms   837
threshold 0.10:  caught  859 / 2005 defaulters   |   missed 1146   |   false alarms  2870
threshold 0.05:  caught 1551 / 2005 defaulters   |   missed  454   |   false alarms 14628


In [24]:
final_threshold = 0.10
final_preds = (logreg_probs >= final_threshold).astype(int)

print("Final model — logistic regression at threshold", final_threshold)
print()
print(confusion_matrix(y_test, final_preds))
print()
print(classification_report(y_test, final_preds))

Final model — logistic regression at threshold 0.1

[[25125  2870]
 [ 1146   859]]

              precision    recall  f1-score   support

           0       0.96      0.90      0.93     27995
           1       0.23      0.43      0.30      2005

    accuracy                           0.87     30000
   macro avg       0.59      0.66      0.61     30000
weighted avg       0.91      0.87      0.88     30000



In [25]:
X.columns

Index(['RevolvingUtilizationOfUnsecuredLines', 'age',
       'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome',
       'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate',
       'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse',
       'NumberOfDependents'],
      dtype='str')

In [27]:
feature_names = X.columns
coefficients = logreg.coef_[0]

coef_table = pd.DataFrame({
    'feature': feature_names,
    'weight': coefficients
}).sort_values('weight', ascending=False)

print(coef_table)

                                feature    weight
2  NumberOfTime30-59DaysPastDueNotWorse  2.052000
6               NumberOfTimes90DaysLate  1.929054
9                    NumberOfDependents  0.104472
7          NumberRealEstateLoansOrLines  0.081058
0  RevolvingUtilizationOfUnsecuredLines -0.003713
5       NumberOfOpenCreditLinesAndLoans -0.057749
3                             DebtRatio -0.082973
1                                   age -0.402108
4                         MonthlyIncome -0.429133
8  NumberOfTime60-89DaysPastDueNotWorse -3.823250


In [28]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_probs = rf.predict_proba(X_test)[:,1]

print("Logistic Regression ROC-AUC:", roc_auc_score(y_test, logreg_probs))
print("Random Forest ROC-AUC:", roc_auc_score(y_test, rf_probs))

Logistic Regression ROC-AUC: 0.7143280217031988
Random Forest ROC-AUC: 0.8422727072299606


In [29]:
for threshold in [0.50, 0.30, 0.20, 0.15, 0.10, 0.05]:
    preds = (rf_probs >= threshold).astype(int)
    cm = confusion_matrix(y_test, preds)
    caught = cm[1,1]
    missed = cm[1,0]
    false_alarms = cm[0,1]
    print(f"threshold {threshold:.2f}:  caught {caught:4d} / 2005 defaulters   |   missed {missed:4d}   |   false alarms {false_alarms:5d}")

threshold 0.50:  caught  398 / 2005 defaulters   |   missed 1607   |   false alarms   300
threshold 0.30:  caught  833 / 2005 defaulters   |   missed 1172   |   false alarms  1129
threshold 0.20:  caught 1089 / 2005 defaulters   |   missed  916   |   false alarms  2095
threshold 0.15:  caught 1233 / 2005 defaulters   |   missed  772   |   false alarms  2992
threshold 0.10:  caught 1413 / 2005 defaulters   |   missed  592   |   false alarms  4659
threshold 0.05:  caught 1647 / 2005 defaulters   |   missed  358   |   false alarms  8356


In [30]:
final_threshold = 0.10
final_preds = (rf_probs >= final_threshold).astype(int)

print("Final model — random forest at threshold", final_threshold)
print()
print(confusion_matrix(y_test, final_preds))
print()
print(classification_report(y_test, final_preds))

Final model — random forest at threshold 0.1

[[23336  4659]
 [  592  1413]]

              precision    recall  f1-score   support

           0       0.98      0.83      0.90     27995
           1       0.23      0.70      0.35      2005

    accuracy                           0.82     30000
   macro avg       0.60      0.77      0.62     30000
weighted avg       0.93      0.82      0.86     30000



In [31]:
feature_names = X.columns
importance = rf.feature_importances_

importances = pd.DataFrame({
    'feature': feature_names,
    'importance': importance
}).sort_values('importance', ascending=False)

print(importances)

                                feature  importance
0  RevolvingUtilizationOfUnsecuredLines    0.192708
3                             DebtRatio    0.175751
4                         MonthlyIncome    0.147640
1                                   age    0.129210
6               NumberOfTimes90DaysLate    0.093566
5       NumberOfOpenCreditLinesAndLoans    0.090131
2  NumberOfTime30-59DaysPastDueNotWorse    0.048562
8  NumberOfTime60-89DaysPastDueNotWorse    0.046387
9                    NumberOfDependents    0.042024
7          NumberRealEstateLoansOrLines    0.034021


In [32]:
from sklearn.ensemble import HistGradientBoostingClassifier

gb = HistGradientBoostingClassifier(random_state=42)
gb.fit(X_train, y_train)
gb_probs = gb.predict_proba(X_test)[:,1]

print("Logistic Regression ROC-AUC:", roc_auc_score(y_test, logreg_probs))
print("Random Forest ROC-AUC:      ", roc_auc_score(y_test, rf_probs))
print("Gradient Boosting ROC-AUC:  ", roc_auc_score(y_test, gb_probs))

Logistic Regression ROC-AUC: 0.7143280217031988
Random Forest ROC-AUC:       0.8422727072299606
Gradient Boosting ROC-AUC:   0.8682134100362596


In [33]:
for threshold in [0.50, 0.30, 0.20, 0.15, 0.10, 0.05]:
    preds = (gb_probs >= threshold).astype(int)
    cm = confusion_matrix(y_test, preds)
    caught = cm[1,1]
    missed = cm[1,0]
    false_alarms = cm[0,1]
    print(f"threshold {threshold:.2f}:  caught {caught:4d} / 2005 defaulters   |   missed {missed:4d}   |   false alarms {false_alarms:5d}")

threshold 0.50:  caught  379 / 2005 defaulters   |   missed 1626   |   false alarms   260
threshold 0.30:  caught  792 / 2005 defaulters   |   missed 1213   |   false alarms   930
threshold 0.20:  caught 1048 / 2005 defaulters   |   missed  957   |   false alarms  1652
threshold 0.15:  caught 1186 / 2005 defaulters   |   missed  819   |   false alarms  2287
threshold 0.10:  caught 1379 / 2005 defaulters   |   missed  626   |   false alarms  3690
threshold 0.05:  caught 1671 / 2005 defaulters   |   missed  334   |   false alarms  7352


In [34]:
final_threshold = 0.10
final_preds = (gb_probs >= final_threshold).astype(int)

print("Final model — gradient boosting at threshold", final_threshold)
print()
print(confusion_matrix(y_test, final_preds))
print()
print(classification_report(y_test, final_preds))

Final model — gradient boosting at threshold 0.1

[[24305  3690]
 [  626  1379]]

              precision    recall  f1-score   support

           0       0.97      0.87      0.92     27995
           1       0.27      0.69      0.39      2005

    accuracy                           0.86     30000
   macro avg       0.62      0.78      0.65     30000
weighted avg       0.93      0.86      0.88     30000



In [35]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    gb, X_test, y_test,
    scoring='roc_auc',
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

perm_table = pd.DataFrame({
    'feature': X.columns,
    'importance': perm.importances_mean
}).sort_values('importance', ascending=False)

print(perm_table)

                                feature  importance
0  RevolvingUtilizationOfUnsecuredLines    0.072122
2  NumberOfTime30-59DaysPastDueNotWorse    0.032563
6               NumberOfTimes90DaysLate    0.032420
8  NumberOfTime60-89DaysPastDueNotWorse    0.014413
1                                   age    0.011756
5       NumberOfOpenCreditLinesAndLoans    0.005633
3                             DebtRatio    0.004882
4                         MonthlyIncome    0.003046
7          NumberRealEstateLoansOrLines    0.002333
9                    NumberOfDependents    0.000289


In [36]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    rf, X_test, y_test,
    scoring='roc_auc',
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

perm_table = pd.DataFrame({
    'feature': X.columns,
    'importance': perm.importances_mean
}).sort_values('importance', ascending=False)

print(perm_table)

                                feature  importance
0  RevolvingUtilizationOfUnsecuredLines    0.063301
6               NumberOfTimes90DaysLate    0.038994
2  NumberOfTime30-59DaysPastDueNotWorse    0.037515
8  NumberOfTime60-89DaysPastDueNotWorse    0.019922
1                                   age    0.011116
4                         MonthlyIncome    0.007258
3                             DebtRatio    0.006110
5       NumberOfOpenCreditLinesAndLoans    0.003707
9                    NumberOfDependents    0.001026
7          NumberRealEstateLoansOrLines    0.001014


In [37]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    logreg, X_test, y_test,
    scoring='roc_auc',
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

perm_table = pd.DataFrame({
    'feature': X.columns,
    'importance': perm.importances_mean
}).sort_values('importance', ascending=False)

print(perm_table)

                                feature  importance
2  NumberOfTime30-59DaysPastDueNotWorse    0.125120
6               NumberOfTimes90DaysLate    0.082302
1                                   age    0.072387
4                         MonthlyIncome    0.008727
9                    NumberOfDependents    0.005222
7          NumberRealEstateLoansOrLines    0.002097
0  RevolvingUtilizationOfUnsecuredLines    0.000024
3                             DebtRatio   -0.000314
5       NumberOfOpenCreditLinesAndLoans   -0.000365
8  NumberOfTime60-89DaysPastDueNotWorse   -0.070266
